# Análisis de Datos por Departamento — Colombia (4 años)

**Flujo del notebook:**
1. Configurar nombres de columnas
2. Subir los 4 CSV (2021–2024)
3. Seleccionar columnas a conservar
4. Unir DataFrames y crear variables nuevas
5. Estadística descriptiva y visualizaciones
6. Exportar resultado final

In [ ]:
# ── Celda 1: Librerías ─────────────────────────────────────────────────────────
!pip install ipywidgets -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
import io, warnings
from google.colab import files
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
print('✓ Librerías cargadas')

---
## ⚙️ Celda 2 — CONFIGURACIÓN
**Edite aquí los nombres de las columnas según sus archivos CSV antes de continuar.**

In [ ]:
# ── Celda 2: Configuración ─────────────────────────────────────────────────────
#
# COLUMNA_DEPARTAMENTO: columna que identifica el departamento.
#   Si es numérico usa el código DANE (ej. 5 = Antioquia, 11 = Bogotá D.C.)
#   Si es texto usa el nombre del departamento (ej. 'ANTIOQUIA')
#
# Ejemplos comunes en datos ICFES / Saber Pro:
#   Código DANE  → 'ESTU_COD_DEPTO_PRESENTACION'
#   Nombre texto → 'ESTU_DEPTO_PRESENTACION'

COLUMNA_DEPARTAMENTO = 'ESTU_COD_DEPTO_PRESENTACION'   # ← CAMBIAR SI ES NECESARIO

# COLUMNA_RESULTADO: columna con el puntaje / resultado del examen.
# Ejemplos: 'MOD_RAZONA_CUANTITAT_PUNT', 'PUNT_GLOBAL', 'PUNTAJE_TOTAL'

COLUMNA_RESULTADO = 'MOD_RAZONA_CUANTITAT_PUNT'        # ← CAMBIAR SI ES NECESARIO

print(f'Columna departamento : {COLUMNA_DEPARTAMENTO}')
print(f'Columna resultado    : {COLUMNA_RESULTADO}')

---
## 📂 Celda 3 — Subir archivos CSV
Suba los 4 archivos **en orden**: 2021 → 2022 → 2023 → 2024.

In [ ]:
# ── Celda 3: Carga de archivos ─────────────────────────────────────────────────
print('📂 Seleccione y suba sus 4 archivos CSV (orden: 2021, 2022, 2023, 2024)')
print('-' * 60)
uploaded = files.upload()
archivos = list(uploaded.keys())

print(f'\n✓ {len(archivos)} archivo(s) cargado(s):')
for i, nombre in enumerate(archivos):
    print(f'   {i+1}. {nombre}')

if len(archivos) < 4:
    print(f'\n⚠ Se esperaban 4 archivos, se cargaron {len(archivos)}.')
    print('  Puede continuar con menos archivos pero las variables de año estarán incompletas.')

---
## 📋 Celda 4 — Crear DataFrames individuales por año

In [ ]:
# ── Celda 4: DataFrames individuales ───────────────────────────────────────────
#
# Los años se asignan en el mismo orden en que subió los archivos.
# Si los subió en un orden diferente, edite la lista 'años_orden'.

años_orden = [2021, 2022, 2023, 2024]   # ← CAMBIAR SI EL ORDEN FUE DISTINTO

def leer_csv(contenido):
    for enc in ['utf-8', 'latin1', 'utf-8-sig', 'cp1252']:
        try:
            return pd.read_csv(io.BytesIO(contenido), low_memory=False, encoding=enc)
        except Exception:
            pass
    raise ValueError('No se pudo leer el archivo con ninguna codificación conocida.')

dataframes = {}
for i, archivo in enumerate(archivos):
    año = años_orden[i] if i < len(años_orden) else 9000 + i
    df = leer_csv(uploaded[archivo])
    df['AÑO'] = año
    dataframes[año] = df
    print(f'✓ df_{año}: {df.shape[0]:>8,} filas × {df.shape[1]:>4} columnas  ← {archivo}')

# Variables individuales accesibles por nombre
df_2021 = dataframes.get(2021)
df_2022 = dataframes.get(2022)
df_2023 = dataframes.get(2023)
df_2024 = dataframes.get(2024)

print('\n✓ DataFrames individuales creados: df_2021, df_2022, df_2023, df_2024')

---
## 🔧 Celda 5 — Seleccionar columnas a conservar
Use **Ctrl + Click** para selección múltiple, o presione los botones de acceso rápido.

In [ ]:
# ── Celda 5: Selector de columnas (widget interactivo) ─────────────────────────
df_ref = list(dataframes.values())[0]
todas_las_columnas = [c for c in df_ref.columns if c != 'AÑO']

print(f'Total de columnas disponibles: {len(todas_las_columnas)}')
print('─' * 50)
for i, col in enumerate(todas_las_columnas):
    print(f'  {i:>4}. {col}')

# ── Widget ─────────────────────────────────────────────────────────────────────
selector = widgets.SelectMultiple(
    options=todas_las_columnas,
    value=todas_las_columnas,
    description='Columnas:',
    layout=widgets.Layout(width='90%', height='280px'),
    style={'description_width': 'initial'}
)

btn_todas   = widgets.Button(description='Conservar TODAS',  button_style='info',    icon='check')
btn_minimas = widgets.Button(description='Solo mínimas',     button_style='warning', icon='minus')
btn_ok      = widgets.Button(description='✔ Confirmar',      button_style='success')
output_msg  = widgets.Output()

estado = {'columnas': list(todas_las_columnas)}

def _btn_todas(b):
    selector.value = tuple(todas_las_columnas)
    with output_msg:
        output_msg.clear_output()
        print(f'✓ Se conservarán TODAS las {len(todas_las_columnas)} columnas')

def _btn_minimas(b):
    cols_min = [c for c in [COLUMNA_DEPARTAMENTO, COLUMNA_RESULTADO] if c in todas_las_columnas]
    selector.value = tuple(cols_min)
    with output_msg:
        output_msg.clear_output()
        print(f'✓ Columnas mínimas seleccionadas: {cols_min}')

def _btn_ok(b):
    sel = list(selector.value) if selector.value else list(todas_las_columnas)
    # Garantizar columnas necesarias para el análisis
    for req in [COLUMNA_DEPARTAMENTO, COLUMNA_RESULTADO]:
        if req in todas_las_columnas and req not in sel:
            sel.append(req)
    estado['columnas'] = sel
    with output_msg:
        output_msg.clear_output()
        print(f'✓ Confirmadas {len(sel)} columnas:')
        print(f'  {sel}')

btn_todas.on_click(_btn_todas)
btn_minimas.on_click(_btn_minimas)
btn_ok.on_click(_btn_ok)

display(widgets.VBox([
    widgets.HBox([btn_todas, btn_minimas, btn_ok]),
    selector,
    output_msg
]))

---
## 🔗 Celda 6 — Unir los 4 DataFrames
Ejecute **después** de confirmar la selección de columnas.

In [ ]:
# ── Celda 6: Unión de los 4 DataFrames ────────────────────────────────────────
cols_keep = estado['columnas'] + ['AÑO']

def filtrar(df, cols):
    return df[[c for c in cols if c in df.columns]].copy()

piezas = [filtrar(df, cols_keep) for df in dataframes.values()]
df_total = pd.concat(piezas, ignore_index=True)

print(f'✓ DataFrame unificado: {df_total.shape[0]:,} filas × {df_total.shape[1]} columnas')
print(f'  Columnas: {list(df_total.columns)}')
df_total.head(3)

---
## ➕ Celda 7 — Crear variables nuevas
Se crean cuatro variables:

| Variable | Descripción |
|---|---|
| `DUMMY_POST` | 0 si el registro es de 2021 o 2022, 1 si es de 2023 o 2024 |
| `NUM_DEPTO` | Número del departamento (1–32), **excluye Bogotá D.C.** |
| `DIST_BOGOTA_KM` | Distancia por carretera desde la capital del depto hasta Bogotá |
| `PROM_RESULTADO_DEPTO` | Promedio del resultado del examen para todos los registros del departamento |

In [ ]:
# ── Celda 7: Creación de variables nuevas ─────────────────────────────────────

# ── 7.1  DUMMY_POST ───────────────────────────────────────────────────────────
# 0 = Pre (2021 / 2022)   |   1 = Post (2023 / 2024)
df_total['DUMMY_POST'] = df_total['AÑO'].apply(lambda x: 0 if x in [2021, 2022] else 1)
print('✓ DUMMY_POST (0 = 2021-2022 / 1 = 2023-2024):')
print(df_total['DUMMY_POST'].value_counts().rename(index={0: 'Pre  (2021-2022)', 1: 'Post (2023-2024)'})
      .to_string())

# ── 7.2  NUM_DEPTO  ───────────────────────────────────────────────────────────
# Referencia de departamentos colombianos (32 deptos, Bogotá D.C. excluida)
# Código DANE → número 1-32
MAPA_DANE = {
     91: 1,   #  1 Amazonas
      5: 2,   #  2 Antioquia
     81: 3,   #  3 Arauca
      8: 4,   #  4 Atlántico
     13: 5,   #  5 Bolívar
     15: 6,   #  6 Boyacá
     17: 7,   #  7 Caldas
     18: 8,   #  8 Caquetá
     85: 9,   #  9 Casanare
     19: 10,  # 10 Cauca
     20: 11,  # 11 Cesar
     27: 12,  # 12 Chocó
     23: 13,  # 13 Córdoba
     25: 14,  # 14 Cundinamarca
     94: 15,  # 15 Guainía
     95: 16,  # 16 Guaviare
     41: 17,  # 17 Huila
     44: 18,  # 18 La Guajira
     47: 19,  # 19 Magdalena
     50: 20,  # 20 Meta
     52: 21,  # 21 Nariño
     54: 22,  # 22 Norte de Santander
     86: 23,  # 23 Putumayo
     63: 24,  # 24 Quindío
     66: 25,  # 25 Risaralda
     88: 26,  # 26 San Andrés y Providencia
     68: 27,  # 27 Santander
     70: 28,  # 28 Sucre
     73: 29,  # 29 Tolima
     76: 30,  # 30 Valle del Cauca
     97: 31,  # 31 Vaupés
     99: 32,  # 32 Vichada
     11: np.nan,  # Bogotá D.C. — EXCLUIDO
}

# Nombre en texto → número 1-32
MAPA_NOMBRE = {
    'AMAZONAS': 1, 'ANTIOQUIA': 2, 'ARAUCA': 3, 'ATLANTICO': 4, 'ATLÁNTICO': 4,
    'BOLIVAR': 5, 'BOLÍVAR': 5, 'BOYACA': 6, 'BOYACÁ': 6,
    'CALDAS': 7, 'CAQUETA': 8, 'CAQUETÁ': 8, 'CASANARE': 9,
    'CAUCA': 10, 'CESAR': 11, 'CHOCO': 12, 'CHOCÓ': 12,
    'CORDOBA': 13, 'CÓRDOBA': 13, 'CUNDINAMARCA': 14,
    'GUAINIA': 15, 'GUAINÍA': 15, 'GUAVIARE': 16,
    'HUILA': 17, 'LA GUAJIRA': 18, 'GUAJIRA': 18,
    'MAGDALENA': 19, 'META': 20,
    'NARINO': 21, 'NARIÑO': 21,
    'NORTE DE SANTANDER': 22, 'PUTUMAYO': 23,
    'QUINDIO': 24, 'QUINDÍO': 24, 'RISARALDA': 25,
    'SAN ANDRES': 26, 'SAN ANDRÉS': 26,
    'SAN ANDRES Y PROVIDENCIA': 26, 'SAN ANDRÉS Y PROVIDENCIA': 26,
    'SANTANDER': 27, 'SUCRE': 28, 'TOLIMA': 29,
    'VALLE DEL CAUCA': 30, 'VALLE': 30,
    'VAUPES': 31, 'VAUPÉS': 31, 'VICHADA': 32,
    # Bogotá D.C. en sus distintas variantes → EXCLUIDA
    'BOGOTA': np.nan, 'BOGOTÁ': np.nan,
    'BOGOTA D.C.': np.nan, 'BOGOTÁ D.C.': np.nan,
    'DISTRITO CAPITAL': np.nan, 'BOGOTA D.C': np.nan,
}

col = df_total[COLUMNA_DEPARTAMENTO]
if pd.api.types.is_numeric_dtype(col):
    df_total['NUM_DEPTO'] = col.map(MAPA_DANE)
    tipo_mapa = 'código DANE numérico'
else:
    col_norm = (col.astype(str).str.upper().str.strip()
                   .str.replace('[ÁÀÂÄ]', 'Á', regex=True)
                   .str.replace('[ÉÈÊË]', 'É', regex=True)
                   .str.replace('[ÍÌÎÏ]', 'Í', regex=True)
                   .str.replace('[ÓÒÔÖ]', 'Ó', regex=True)
                   .str.replace('[ÚÙÛÜ]', 'Ú', regex=True))
    df_total['NUM_DEPTO'] = col_norm.map(MAPA_NOMBRE)
    tipo_mapa = 'nombre de departamento'

print(f'\n✓ NUM_DEPTO creada (método: {tipo_mapa})')
print(f'  Registros sin mapeo (incl. Bogotá D.C.): {df_total["NUM_DEPTO"].isna().sum():,}')

# ── 7.3  DIST_BOGOTA_KM ──────────────────────────────────────────────────────
# Distancia por carretera desde la capital departamental hasta Bogotá (km)
# Fuente: estimaciones estándar de distancias viales Colombia
DIST_BOGOTA = {
     1: 1500,  #  1 Amazonas    — Leticia         (ruta mixta)
     2:  415,  #  2 Antioquia   — Medellín
     3:  800,  #  3 Arauca      — Arauca
     4: 1002,  #  4 Atlántico   — Barranquilla
     5: 1044,  #  5 Bolívar     — Cartagena
     6:  148,  #  6 Boyacá      — Tunja
     7:  308,  #  7 Caldas      — Manizales
     8:  487,  #  8 Caquetá     — Florencia
     9:  360,  #  9 Casanare    — Yopal
    10:  594,  # 10 Cauca       — Popayán
    11: 1011,  # 11 Cesar       — Valledupar
    12:  529,  # 12 Chocó       — Quibdó
    13:  821,  # 13 Córdoba     — Montería
    14:    0,  # 14 Cundinamarca— Bogotá (capital departamental = Bogotá)
    15: 1800,  # 15 Guainía     — Inírida         (ruta mixta)
    16:  679,  # 16 Guaviare    — San José del Guaviare
    17:  303,  # 17 Huila       — Neiva
    18: 1204,  # 18 La Guajira  — Riohacha
    19: 1096,  # 19 Magdalena   — Santa Marta
    20:   89,  # 20 Meta        — Villavicencio
    21:  869,  # 21 Nariño      — Pasto
    22:  605,  # 22 Nte. Santander — Cúcuta
    23:  612,  # 23 Putumayo    — Mocoa
    24:  291,  # 24 Quindío     — Armenia
    25:  341,  # 25 Risaralda   — Pereira
    26:  752,  # 26 San Andrés  — San Andrés (distancia aérea)
    27:  396,  # 27 Santander   — Bucaramanga
    28:  942,  # 28 Sucre       — Sincelejo
    29:  204,  # 29 Tolima      — Ibagué
    30:  462,  # 30 Valle Cauca — Cali
    31: 1600,  # 31 Vaupés      — Mitú             (ruta mixta)
    32: 1395,  # 32 Vichada     — Puerto Carreño
}
df_total['DIST_BOGOTA_KM'] = df_total['NUM_DEPTO'].map(DIST_BOGOTA)
print('\n✓ DIST_BOGOTA_KM creada')

# ── 7.4  PROM_RESULTADO_DEPTO ─────────────────────────────────────────────────
# Media del resultado del examen por departamento, asignada a cada registro
df_total['PROM_RESULTADO_DEPTO'] = (
    df_total.groupby(COLUMNA_DEPARTAMENTO)[COLUMNA_RESULTADO]
            .transform('mean')
)
print('✓ PROM_RESULTADO_DEPTO creada')

# ── Resumen ───────────────────────────────────────────────────────────────────
print('\n' + '─'*55)
print(f'{"Variable":<25} {"No nulos":>12} {"Nulos":>10}')
print('─'*55)
for v in ['DUMMY_POST', 'NUM_DEPTO', 'DIST_BOGOTA_KM', 'PROM_RESULTADO_DEPTO']:
    nn = df_total[v].notna().sum()
    nl = df_total[v].isna().sum()
    print(f'{v:<25} {nn:>12,} {nl:>10,}')

---
## 🗑️ Celda 8 — Excluir Bogotá D.C. y registros sin mapeo

In [ ]:
# ── Celda 8: Excluir Bogotá D.C. ──────────────────────────────────────────────
n_antes = len(df_total)
df_total = df_total[df_total['NUM_DEPTO'].notna()].copy()
df_total['NUM_DEPTO'] = df_total['NUM_DEPTO'].astype(int)
n_despues = len(df_total)

print(f'Registros eliminados (Bogotá D.C. + sin mapeo): {n_antes - n_despues:,}')
print(f'Registros finales para análisis               : {n_despues:,}')

# Tabla de referencia: número ↔ nombre ↔ distancia
NOMBRES_DEPTO = {
     1: 'Amazonas',         2: 'Antioquia',           3: 'Arauca',
     4: 'Atlántico',        5: 'Bolívar',              6: 'Boyacá',
     7: 'Caldas',           8: 'Caquetá',              9: 'Casanare',
    10: 'Cauca',           11: 'Cesar',               12: 'Chocó',
    13: 'Córdoba',         14: 'Cundinamarca',        15: 'Guainía',
    16: 'Guaviare',        17: 'Huila',               18: 'La Guajira',
    19: 'Magdalena',       20: 'Meta',                21: 'Nariño',
    22: 'Norte de Santander', 23: 'Putumayo',         24: 'Quindío',
    25: 'Risaralda',       26: 'San Andrés y Prov.',  27: 'Santander',
    28: 'Sucre',           29: 'Tolima',              30: 'Valle del Cauca',
    31: 'Vaupés',          32: 'Vichada',
}
df_total['NOMBRE_DEPTO'] = df_total['NUM_DEPTO'].map(NOMBRES_DEPTO)

print('\n✓ Tabla de referencia departamentos:')
ref = pd.DataFrame([
    {'Num': k, 'Departamento': NOMBRES_DEPTO[k], 'Capital': cap, 'Dist_Bogotá_km': DIST_BOGOTA[k]}
    for k, cap in [
        (1,'Leticia'),(2,'Medellín'),(3,'Arauca'),(4,'Barranquilla'),(5,'Cartagena'),
        (6,'Tunja'),(7,'Manizales'),(8,'Florencia'),(9,'Yopal'),(10,'Popayán'),
        (11,'Valledupar'),(12,'Quibdó'),(13,'Montería'),(14,'Bogotá'),(15,'Inírida'),
        (16,'S.J. Guaviare'),(17,'Neiva'),(18,'Riohacha'),(19,'Santa Marta'),(20,'Villavicencio'),
        (21,'Pasto'),(22,'Cúcuta'),(23,'Mocoa'),(24,'Armenia'),(25,'Pereira'),
        (26,'San Andrés'),(27,'Bucaramanga'),(28,'Sincelejo'),(29,'Ibagué'),(30,'Cali'),
        (31,'Mitú'),(32,'Puerto Carreño'),
    ]
]).set_index('Num')
display(ref)

---
## 📊 Celda 9 — Estadística descriptiva

In [ ]:
# ── Celda 9: Estadística descriptiva ──────────────────────────────────────────

vars_analisis = [COLUMNA_RESULTADO, 'DUMMY_POST', 'NUM_DEPTO',
                 'DIST_BOGOTA_KM', 'PROM_RESULTADO_DEPTO']
vars_ok = [v for v in vars_analisis if v in df_total.columns]

print('=' * 70)
print('  ESTADÍSTICA DESCRIPTIVA — VARIABLES PRINCIPALES')
print('=' * 70)
display(df_total[vars_ok].describe().round(3))

# ── Distribución por año ──────────────────────────────────────────────────────
print('\n--- Registros por año ---')
display(df_total['AÑO'].value_counts().sort_index().rename('N registros').to_frame())

# ── Distribución por periodo ──────────────────────────────────────────────────
print('\n--- Registros por periodo (DUMMY_POST) ---')
display(df_total['DUMMY_POST']
        .value_counts()
        .rename(index={0: 'Pre  (2021-2022)', 1: 'Post (2023-2024)'})
        .rename('N registros')
        .to_frame())

# ── Promedio del resultado por departamento ───────────────────────────────────
print('\n--- Promedio del resultado por departamento ---')
tabla = (df_total
         .groupby(['NUM_DEPTO', 'NOMBRE_DEPTO'])[COLUMNA_RESULTADO]
         .agg(Promedio='mean', Desv_Est='std', N='count')
         .round(3)
         .reset_index()
         .sort_values('NUM_DEPTO')
         .set_index('NUM_DEPTO'))
display(tabla)

---
## 📈 Celda 10 — Visualizaciones

In [ ]:
# ── Celda 10: Visualizaciones ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Análisis Descriptivo — Resultados del Examen por Departamento',
             fontsize=15, fontweight='bold')

media_gral = df_total[COLUMNA_RESULTADO].mean()

# ── Panel 1: Histograma del puntaje ───────────────────────────────────────────
ax = axes[0, 0]
ax.hist(df_total[COLUMNA_RESULTADO].dropna(), bins=60,
        color='#2196F3', edgecolor='white', alpha=0.85)
ax.axvline(media_gral, color='red', linestyle='--', linewidth=2,
           label=f'Media general: {media_gral:.1f}')
ax.set_title('Distribución del Resultado del Examen')
ax.set_xlabel('Puntaje')
ax.set_ylabel('Frecuencia')
ax.legend()

# ── Panel 2: Boxplot por año ───────────────────────────────────────────────────
ax = axes[0, 1]
grupos_año = [df_total.loc[df_total['AÑO'] == a, COLUMNA_RESULTADO].dropna().values
              for a in sorted(df_total['AÑO'].unique())]
bp = ax.boxplot(grupos_año, patch_artist=True,
                labels=sorted(df_total['AÑO'].unique()),
                medianprops=dict(color='black', linewidth=2))
colores_bp = ['#90CAF9', '#42A5F5', '#FB8C00', '#EF5350']
for patch, color in zip(bp['boxes'], colores_bp[:len(grupos_año)]):
    patch.set_facecolor(color)
ax.set_title('Resultado por Año')
ax.set_xlabel('Año')
ax.set_ylabel('Puntaje')

# ── Panel 3: Promedio por departamento (barra horizontal) ─────────────────────
ax = axes[1, 0]
prom_depto = (df_total.groupby(['NUM_DEPTO', 'NOMBRE_DEPTO'])[COLUMNA_RESULTADO]
              .mean().reset_index().sort_values(COLUMNA_RESULTADO))
colores_bar = ['#EF5350' if v < media_gral else '#66BB6A'
               for v in prom_depto[COLUMNA_RESULTADO]]
ax.barh(prom_depto['NOMBRE_DEPTO'], prom_depto[COLUMNA_RESULTADO],
        color=colores_bar, alpha=0.85, edgecolor='white')
ax.axvline(media_gral, color='black', linestyle='--', linewidth=1.2,
           label=f'Media: {media_gral:.1f}')
ax.set_title('Promedio por Departamento\n(rojo = bajo la media nacional)')
ax.set_xlabel('Promedio')
ax.tick_params(axis='y', labelsize=7)
ax.legend(fontsize=9)

# ── Panel 4: Dispersión distancia vs promedio ─────────────────────────────────
ax = axes[1, 1]
agg = (df_total.groupby('NUM_DEPTO')
       .agg(promedio=(COLUMNA_RESULTADO, 'mean'),
            distancia=('DIST_BOGOTA_KM', 'first'),
            nombre=('NOMBRE_DEPTO', 'first'))
       .reset_index())
ax.scatter(agg['distancia'], agg['promedio'],
           s=80, color='#7B1FA2', alpha=0.8, edgecolors='white', linewidth=0.5)
# Tendencia
mask = agg['distancia'].notna() & agg['promedio'].notna()
if mask.sum() > 1:
    z = np.polyfit(agg.loc[mask, 'distancia'], agg.loc[mask, 'promedio'], 1)
    p = np.poly1d(z)
    xs = np.linspace(agg['distancia'].min(), agg['distancia'].max(), 200)
    ax.plot(xs, p(xs), 'r--', linewidth=1.5, label='Tendencia lineal')
for _, row in agg.iterrows():
    ax.annotate(str(int(row['NUM_DEPTO'])),
                (row['distancia'], row['promedio']),
                fontsize=7, ha='center', va='bottom', color='#333')
ax.set_title('Distancia a Bogotá vs Promedio del Resultado')
ax.set_xlabel('Distancia a Bogotá (km)')
ax.set_ylabel('Promedio del Resultado')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('analisis_departamentos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Gráfico guardado como analisis_departamentos.png')

---
## 💾 Celda 11 — Exportar DataFrame final

In [ ]:
# ── Celda 11: Exportar ────────────────────────────────────────────────────────
nombre_salida = 'datos_analisis_departamentos.csv'
df_total.to_csv(nombre_salida, index=False, encoding='utf-8-sig')
files.download(nombre_salida)

print('✓ Proceso completado.')
print(f'  Archivo exportado : {nombre_salida}')
print(f'  Registros finales : {len(df_total):,}')
print(f'  Columnas totales  : {df_total.shape[1]}')
print('\nVariables nuevas incluidas:')
print('  AÑO                  — Año del registro (2021-2024)')
print('  DUMMY_POST           — 0=Pre (2021-2022) / 1=Post (2023-2024)')
print('  NUM_DEPTO            — Número del departamento (1-32)')
print('  NOMBRE_DEPTO         — Nombre del departamento')
print('  DIST_BOGOTA_KM       — Distancia capital depto → Bogotá (km)')
print('  PROM_RESULTADO_DEPTO — Promedio del resultado para el departamento')